# 02 · Modelado de demoras y entregable

**Objetivo:** predecir si un vuelo se va a demorar (`demora`) con un **Árbol de Decisión** y **KNN**, compararlos
por métricas y por **costo en USD**, y usar el árbol para predecir los casos nuevos y asignar un beneficio a cada pasajero.

Metodología CRISP-DM: preparación → modelado → evaluación → implementación.

> Los modelos de Regresión Logística y Análisis Discriminante del informe se hicieron en **IBM SPSS**
> (el LDA también en Colab); acá sólo están los que se hicieron en Python (árbol y KNN).

**Datos:** `Vuelos.csv`, `casosNuevos.csv`, `casosNuevos_Clientes.csv`, `Clientes.csv`.

**Costos de los errores** (informe, 6.1.1): falso positivo = **80 USD** (beneficio entregado sin necesidad);
falso negativo = **50 USD** (demora no anticipada).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.tree import plot_tree

from vuelos_ds import config, data, modeling, plots

plots.set_style()

## Fase 3 · Preparación de los datos

In [ ]:
df = data.load_vuelos()
casos_nuevos = data.load_casos_nuevos()

pre = modeling.FlightPreprocessor()
df_prep = pre.fit_transform(df)
display(df_prep.head())

X_train, X_test, y_train, y_test = modeling.split_train_test(df_prep)
print(f"Entrenamiento: {len(X_train)} | Test: {len(X_test)}")

## Fase 4 · Modelado

### 4.1 Árbol de Decisión: búsqueda de la mejor configuración

Se prueban profundidades, criterio (Gini / entropía) y con o sin balanceo de clases, y se calcula el costo de
cada configuración.

> **Ojo:** igual que en el informe, se evalúa sobre el conjunto de prueba. Elegir hiperparámetros con el mismo
> conjunto con el que después se reporta el resultado es optimista. Lo más correcto sería validación cruzada
> o un conjunto de validación aparte.

In [ ]:
grilla = modeling.buscar_arbol(X_train, y_train, X_test, y_test)
costo_base = modeling.costo_baseline(y_test)

print(f"Costo de NO predecir nunca una demora (baseline): {costo_base:,} USD")
display(grilla.sort_values("costo").round(4).reset_index(drop=True))

**Configuración elegida (informe): entropía, profundidad 5, sin balancear.** Las profundidades 3 y 4 tienen menor
costo pero un recall demasiado bajo (casi no detectan demoras); la profundidad 5 es el mejor equilibrio.

In [ ]:
arbol = modeling.train_tree(X_train, y_train)

plt.figure(figsize=(30, 12))
plot_tree(arbol, feature_names=list(X_train.columns), class_names=["No Demora", "Demora"],
          filled=True, rounded=True, fontsize=8)
plt.title("Estructura del Árbol de Decisión")
plt.show()

### 4.2 KNN: elección de K

In [ ]:
knn_k = modeling.buscar_knn(X_train, y_train, X_test, y_test)
display(knn_k.round(4))

Al subir K mejora levemente el accuracy pero cae el recall (se detectan menos demoras). Se elige un valor
intermedio: **K = 5**. KNN incluye el `StandardScaler` en su pipeline.

In [ ]:
knn = modeling.train_knn(X_train, y_train, n_neighbors=5)

## Fase 5 · Evaluación

In [ ]:
resultados = pd.DataFrame([
    modeling.evaluate(arbol, X_test, y_test, "Árbol de Decisión"),
    modeling.evaluate(knn, X_test, y_test, "KNN (k=5)"),
]).set_index("modelo")

baseline = (y_test == 0).mean()
print(f"\nBaseline (predecir siempre 'no demora'): accuracy = {baseline:.3f} | costo = {costo_base:,} USD")
display(resultados.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plots.plot_confusion(arbol, X_test, y_test, "Árbol", ax=axes[0])
plots.plot_confusion(knn, X_test, y_test, "KNN", ax=axes[1])
plt.tight_layout()
plt.show()

**Lectura de resultados** *(corrida del informe con los datos reales)*

| Modelo | Accuracy | Precisión | Recall | Falsos + | Falsos − | Costo |
|---|---|---|---|---|---|---|
| Árbol (entropía, prof. 5) | 0,7084 | 0,5935 | 0,2680 | 263 | 1.049 | 73.490 USD |
| KNN (k = 5) | 0,6564 | 0,4369 | 0,2659 | 491 | 1.052 | 91.880 USD |
| *Baseline: nunca predecir demora* | 0,6816 | — | 0 | 0 | 1.433 | *71.650 USD* |

- El árbol gana a KNN en todas las métricas y cuesta ~18.400 USD menos. Además es interpretable.
- **Con esta estructura de costos, ningún modelo supera al baseline.** Marcar un vuelo como demorado ahorra 50 USD
  si acierta y cuesta 80 USD si se equivoca, así que sólo conviene si la precisión supera
  80 / (80 + 50) ≈ 61,5 %. El árbol tiene 59,4 %, por lo que cuesta ~1.840 USD más que no hacer nada.
  Es un punto a discutir (¿el beneficio tiene un valor de fidelización que el modelo de costos no recoge?) y a
  mejorar, por ejemplo **ajustando el umbral de decisión** para subir la precisión.
- El recall es bajo (27 %): el modelo deja pasar casi tres de cada cuatro demoras.

## Fase 6 · Implementación

Se predicen los casos nuevos y se asigna un beneficio a cada pasajero:

| Predicción | ¿Socio del programa de millas? | Beneficio |
|---|---|---|
| Demora | Sí | Sala VIP |
| Demora | No | Voucher comida |
| Sin demora | — | Ninguno |

In [ ]:
predichos = modeling.predecir_casos_nuevos(arbol, pre, casos_nuevos)
final = modeling.armar_entregable(predichos, data.load_relacion(), data.load_clientes())

display(final.head())
print(f"Registros del entregable: {len(final)}")
display(final["Tipo beneficio"].value_counts().to_frame("pasajeros"))

In [ ]:
salida = config.get_output_dir() / "Resultado_Final_Grupo01.xlsx"
final.to_excel(salida, index=False)
print("Archivo generado:", salida)